# 🚀 AI Agent Platform Labs — Environment Setup

This notebook will guide you through deploying all Azure resources needed for Labs 01-07.

## What This Notebook Does

1. **Checks prerequisites** — Python version, Azure CLI, required tools
2. **Deploys Azure resources** — All services via Bicep (one command)
3. **Generates your `.env` file** — All connection strings in one place

**Estimated time:** ~20 minutes

---

## Step 1: Check Prerequisites

In [ ]:
import sys
import subprocess
import shutil

print("🔍 Checking Prerequisites...")
print("=" * 50)

all_ok = True

# Check Python version
py_ver = sys.version_info
py_ok = py_ver >= (3, 11)
print(f"\n📌 Python: {py_ver.major}.{py_ver.minor}.{py_ver.micro}")
print(f"   {'✅ OK' if py_ok else '❌ Need 3.11+'}")
all_ok = all_ok and py_ok

# Check Azure CLI
az_ok = shutil.which("az") is not None
print(f"\n📌 Azure CLI: {'✅ Found' if az_ok else '❌ Not found — install: https://aka.ms/installazurecli'}")
all_ok = all_ok and az_ok

# Check jq
jq_ok = shutil.which("jq") is not None
print(f"\n📌 jq: {'✅ Found' if jq_ok else '❌ Not found — install: brew install jq (macOS) or winget install jqlang.jq (Windows)'}")
all_ok = all_ok and jq_ok

# Check Git
git_ok = shutil.which("git") is not None
print(f"\n📌 Git: {'✅ Found' if git_ok else '❌ Not found'}")
all_ok = all_ok and git_ok

# Check Azure login
if az_ok:
    try:
        result = subprocess.run(["az", "account", "show", "--query", "name", "-o", "tsv"],
                              capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"\n📌 Azure login: ✅ Logged in as subscription: {result.stdout.strip()}")
        else:
            print(f"\n📌 Azure login: ❌ Not logged in — run 'az login' in terminal")
            all_ok = False
    except Exception:
        print(f"\n📌 Azure login: ⚠️ Could not check — run 'az login' in terminal")

print("\n" + "=" * 50)
if all_ok:
    print("🎉 All prerequisites met! Continue to Step 2.")
else:
    print("⚠️  Some prerequisites are missing. Fix them before continuing.")
    print("   See PREREQUISITES.md for installation instructions.")

## Step 2: Deploy Azure Resources

This will deploy all required Azure services using our Bicep template.

**What gets deployed:**
- 🧠 Azure OpenAI (GPT-5.2, GPT-4o-mini, text-embedding-3-large)
- 🔍 Azure AI Search (for RAG in Lab 03)
- 💾 Azure Cosmos DB Serverless (for memory/state in Lab 03)
- 🛡️ Azure AI Content Safety (for guardrails in Lab 05)
- 📦 Storage Account (for documents)

> ⏱️ **This takes about 5-10 minutes.** Go get a coffee! ☕

In [ ]:
import subprocess
import os

# Configuration — change these if you want
RESOURCE_GROUP = "rg-agent-platform-labs"
LOCATION = "swedencentral"  # Best availability for all models
BASE_NAME = "agentlabs"

print(f"📦 Resource Group: {RESOURCE_GROUP}")
print(f"🌍 Location: {LOCATION}")
print(f"📛 Base Name: {BASE_NAME}")
print()

# Create resource group
print("📦 Creating resource group...")
result = subprocess.run(
    ["az", "group", "create", "--name", RESOURCE_GROUP, "--location", LOCATION, "--output", "none"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Resource group created")
else:
    print(f"❌ Failed: {result.stderr}")
    raise Exception("Resource group creation failed")

# Deploy Bicep
infra_dir = os.path.join("..", "..", "infra")
bicep_file = os.path.join(infra_dir, "main.bicep")

print()
print("🔧 Deploying Azure resources (this takes 5-10 minutes)...")
print("   Deploying: Azure OpenAI, AI Search, Cosmos DB, Content Safety, Storage")
print()

result = subprocess.run(
    ["az", "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", bicep_file,
     "--parameters", f"baseName={BASE_NAME}", f"location={LOCATION}",
     "--query", "properties.outputs",
     "--output", "json"],
    capture_output=True, text=True,
    timeout=900  # 15 min timeout
)

if result.returncode == 0:
    print("✅ All resources deployed successfully!")
    deployment_output = result.stdout
else:
    print(f"❌ Deployment failed:")
    print(result.stderr[:1000])
    raise Exception("Deployment failed")

## Step 3: Generate .env File

This extracts all connection strings from the deployment and saves them to `labs/.env`.

In [ ]:
import json
from datetime import datetime

# Parse deployment outputs
outputs = json.loads(deployment_output)

# Get subscription ID
sub_result = subprocess.run(["az", "account", "show", "--query", "id", "-o", "tsv"],
                           capture_output=True, text=True)
subscription_id = sub_result.stdout.strip()

# Generate .env content
env_content = f"""# ===========================================
# AI Agent Platform Labs - Environment Config
# Generated: {datetime.now().isoformat()}
# Region: {LOCATION}
# ===========================================
# ⚠️  This file contains secrets. Never commit it to Git!

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RESOURCE_GROUP}
AZURE_LOCATION={LOCATION}

# ── Azure OpenAI ──────────────────────────────────────
AZURE_OPENAI_ENDPOINT={outputs['aiServicesEndpoint']['value']}
AZURE_OPENAI_API_KEY={outputs['aiServicesKey']['value']}
AZURE_OPENAI_API_VERSION=2024-12-01-preview

# Model deployments
AZURE_OPENAI_DEPLOYMENT_GPT52=gpt-52
AZURE_OPENAI_DEPLOYMENT_GPT4O_MINI=gpt-4o-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# ── Azure AI Search (Lab 03 - RAG) ───────────────────
AZURE_SEARCH_ENDPOINT={outputs['searchServiceEndpoint']['value']}
AZURE_SEARCH_API_KEY={outputs['searchServiceAdminKey']['value']}
AZURE_SEARCH_INDEX_NAME=agent-labs-index

# ── Azure Cosmos DB (Lab 03 - Memory & State) ────────
AZURE_COSMOS_ENDPOINT={outputs['cosmosEndpoint']['value']}
AZURE_COSMOS_KEY={outputs['cosmosKey']['value']}
AZURE_COSMOS_DATABASE={outputs['cosmosDatabaseName']['value']}

# ── Azure AI Content Safety (Lab 05 - Guardrails) ────
AZURE_CONTENT_SAFETY_ENDPOINT={outputs['contentSafetyEndpoint']['value']}
AZURE_CONTENT_SAFETY_KEY={outputs['contentSafetyKey']['value']}

# ── Azure Storage (Lab 03 - Documents) ───────────────
AZURE_STORAGE_CONNECTION_STRING={outputs['storageConnectionString']['value']}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
"""

# Write .env file
env_path = os.path.join("..", ".env")
with open(env_path, 'w') as f:
    f.write(env_content)

print(f"✅ .env file generated at: {os.path.abspath(env_path)}")
print()
print("📋 Resources configured:")
print(f"   🧠 Azure OpenAI:      {outputs['aiServicesEndpoint']['value']}")
print(f"   🔍 Azure AI Search:    {outputs['searchServiceEndpoint']['value']}")
print(f"   💾 Cosmos DB:          {outputs['cosmosEndpoint']['value']}")
print(f"   🛡️ Content Safety:     {outputs['contentSafetyEndpoint']['value']}")

## ✅ Setup Complete!

Your environment is ready. Next steps:

1. **Validate:** Open `health-check.ipynb` and run all cells
2. **Start learning:** Open `../lab-01-react-agent/lab.ipynb`

---

> **[→ Run Health Check](health-check.ipynb)** | **[→ Start Lab 01](../lab-01-react-agent/lab.ipynb)**